# Фильтрация после аннотации — `ERP003950`

Входом служит **AIRR для каждого рида** из этапа аннотации.
Объединённые FASTQ и AIRR обрабатываются строго синхронно.

Входы:
- `merged/fastq/{sample}_assemble-pass.fastq.gz`
- `annotation/igblast/{sample}.airr.tsv`

Выходы:
- `post_annotation_filtered/fastq/{sample}_filtered.fastq.gz`
- `post_annotation_filtered/airr_pass/{sample}.airr.tsv`
- `post_annotation_filtered/airr_reject/{sample}.airr.tsv`
- `post_annotation_filtered/filter_summary.json`


In [ ]:
import csv, gzip, json, os, shutil, time
from collections import Counter
from pathlib import Path

DATASET = "ERP003950"
RAW_SOURCE_DATASET = "ERP003950"

SAMPLES = [
    "ERR346596", "ERR346597", "ERR346598",
    "ERR346599", "ERR346600", "ERR346601",
]

MIN_VJ_LENGTH = 300
MIN_V_IDENTITY = 80.0
MIN_J_IDENTITY = 85.0
MAX_V_SUPPORT = 1e-5
MAX_J_SUPPORT = 1e-5
EXPECTED_LOCUS = {sample: "IGH" for sample in SAMPLES}

FORCE = False

def resolve_volume():
    candidates = []
    if os.environ.get("BCR_VOLUME"):
        candidates.append(Path(os.environ["BCR_VOLUME"]))
    candidates += [
        Path("/data/user/epishkin"),
        Path("/Users/epishkin/workspace/bcr-assembler"),
    ]
    start = Path.cwd().resolve()
    candidates += [start, *start.parents]

    seen = set()
    for root in candidates:
        if str(root) in seen:
            continue
        seen.add(str(root))
        if (root / "results" / DATASET / "merged" / "fastq").is_dir():
            return root
    raise FileNotFoundError(
        f"Cannot locate results/{DATASET}; set BCR_VOLUME explicitly."
    )

VOLUME = resolve_volume()
DATASET_DIR = VOLUME / "results" / DATASET
MERGED_DIR = DATASET_DIR / "merged" / "fastq"
ANNOT_DIR = DATASET_DIR / "annotation" / "igblast"

FINAL_DIR = DATASET_DIR / "post_annotation_filtered"
STAGING_DIR = DATASET_DIR / ".post_annotation_filtered.staging"

found_samples = sorted(
    p.name.removesuffix("_assemble-pass.fastq.gz")
    for p in MERGED_DIR.glob("*_assemble-pass.fastq.gz")
)
if found_samples != SAMPLES:
    raise RuntimeError(
        f"Unexpected merged sample set. Expected {SAMPLES}, found {found_samples}"
    )

print("DATASET_DIR:", DATASET_DIR)
print("MERGED_DIR:", MERGED_DIR)
print("ANNOT_DIR:", ANNOT_DIR)
print("FINAL_DIR:", FINAL_DIR)


In [ ]:
TRUE = {"true", "t", "1", "yes", "y"}
FALSE = {"false", "f", "0", "no", "n"}

def as_bool(v):
    x = str(v).strip().lower()
    return True if x in TRUE else False if x in FALSE else None

_COMPLEMENT = str.maketrans(
    "ACGTRYKMSWBDHVN",
    "TGCAYRMKSWVHDBN",
)

def reverse_complement(sequence):
    return str(sequence).upper().translate(_COMPLEMENT)[::-1]

def has_ambiguous_bases(sequence):
    return any(base not in {"A", "C", "G", "T"} for base in str(sequence).upper())

def expected_airr_sequence(query_sequence, row):
    query_sequence = str(query_sequence).upper()
    return (
        reverse_complement(query_sequence)
        if as_bool(row.get("rev_comp")) is True
        else query_sequence
    )

def as_float(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

def as_int(v):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return None

REQUIRED = {
    "sequence_id", "sequence", "locus",
    "productive", "stop_codon", "complete_vdj",
    "v_call", "j_call",
    "v_sequence_start", "j_sequence_end", "v_germline_start",
    "v_identity", "j_identity", "v_support", "j_support",
}

def annotation_path(sample):
    return ANNOT_DIR / f"{sample}.airr.tsv"

def validate_schema(path):
    if not path.exists():
        raise FileNotFoundError(
            f"{path}: annotation AIRR file is missing"
        )
    with open(path, newline="") as fh:
        fields = csv.DictReader(fh, delimiter="\t").fieldnames or []
    missing = sorted(REQUIRED - set(fields))
    if missing:
        raise ValueError(f"{path}: missing AIRR fields {missing}")

for sample in SAMPLES:
    validate_schema(annotation_path(sample))

print("Per-read AIRR schemas OK")


In [ ]:
def evaluate(row, expected_locus):
    reasons = []

    if has_ambiguous_bases(row.get("sequence", "")):
        reasons.append("ambiguous_sequence")

    # 1) Аннотированный участок от V до J.
    v_start = as_int(row.get("v_sequence_start"))
    j_end = as_int(row.get("j_sequence_end"))
    if v_start is None or j_end is None:
        reasons.append("missing_vj_coordinates")
    elif j_end - v_start + 1 < MIN_VJ_LENGTH:
        reasons.append("vj_span_lt_300")

    # 2) Требуем наличие начала germline-выравнивания V.
    if as_int(row.get("v_germline_start")) != 1:
        reasons.append("v_gene_5prime_incomplete")

    # 3) AIRR-поле complete_vdj задаёт критерий полноты 3'-конца J.
    # Не требуем j_sequence_end == len(sequence): корректная последовательность может
    # продолжаться в константную область после J.
    if as_bool(row.get("complete_vdj")) is not True:
        reasons.append("j_3prime_or_vdj_incomplete")

    # 4) ERP003950 — датасет тяжёлых цепей IgG.
    if str(row.get("locus", "")).strip() != expected_locus:
        reasons.append("unexpected_locus")

    # Записи без V/J-вызовов не включаются в набор истины симуляции.
    if not str(row.get("v_call", "")).strip():
        reasons.append("missing_v_call")
    if not str(row.get("j_call", "")).strip():
        reasons.append("missing_j_call")

    # 5) Биологическая продуктивность.
    if as_bool(row.get("productive")) is not True:
        reasons.append("nonproductive")
    if as_bool(row.get("stop_codon")) is not False:
        reasons.append("stop_codon_or_missing")

    # 6) Достоверность аннотации IgBLAST.
    vi = as_float(row.get("v_identity"))
    ji = as_float(row.get("j_identity"))
    vs = as_float(row.get("v_support"))
    js = as_float(row.get("j_support"))

    if vi is None:
        reasons.append("missing_v_identity")
    elif vi < MIN_V_IDENTITY:
        reasons.append("low_v_identity")

    if ji is None:
        reasons.append("missing_j_identity")
    elif ji < MIN_J_IDENTITY:
        reasons.append("low_j_identity")

    if vs is None:
        reasons.append("missing_v_support")
    elif vs > MAX_V_SUPPORT:
        reasons.append("poor_v_support")

    if js is None:
        reasons.append("missing_j_support")
    elif js > MAX_J_SUPPORT:
        reasons.append("poor_j_support")

    return reasons

print({
    "MIN_VJ_LENGTH": MIN_VJ_LENGTH,
    "MIN_V_IDENTITY": MIN_V_IDENTITY,
    "MIN_J_IDENTITY": MIN_J_IDENTITY,
    "MAX_V_SUPPORT": MAX_V_SUPPORT,
    "MAX_J_SUPPORT": MAX_J_SUPPORT,
    "EXPECTED_LOCUS": EXPECTED_LOCUS,
})


In [ ]:
def fastq_records(path):
    with gzip.open(path, "rt") as fh:
        while True:
            header = fh.readline()
            if not header:
                break
            seq = fh.readline()
            plus = fh.readline()
            qual = fh.readline()
            if not qual or not header.startswith("@") or not plus.startswith("+"):
                raise ValueError(f"Malformed FASTQ: {path}")
            yield header, seq, plus, qual

def fastq_id(header):
    return header[1:].strip().split(None, 1)[0]

def filter_sample(sample, out):
    fq = MERGED_DIR / f"{sample}_assemble-pass.fastq.gz"
    airr = annotation_path(sample)

    pass_fastq = out / "fastq" / f"{sample}_filtered.fastq.gz"
    pass_airr = out / "airr_pass" / f"{sample}.airr.tsv"
    reject_airr = out / "airr_reject" / f"{sample}.airr.tsv"

    counts = Counter()
    reason_counts = Counter()
    expected_locus = EXPECTED_LOCUS[sample]

    with open(airr, newline="") as af, \
         gzip.open(pass_fastq, "wt") as ofq, \
         open(pass_airr, "w", newline="") as op, \
         open(reject_airr, "w", newline="") as orej:

        reader = csv.DictReader(af, delimiter="\t")
        fields = reader.fieldnames or []

        pass_writer = csv.DictWriter(
            op, fieldnames=fields, delimiter="\t", lineterminator="\n"
        )
        reject_writer = csv.DictWriter(
            orej,
            fieldnames=fields + ["filter_reasons"],
            delimiter="\t",
            lineterminator="\n",
        )
        pass_writer.writeheader()
        reject_writer.writeheader()

        fq_iter = fastq_records(fq)
        started = time.monotonic()
        last_heartbeat = started
        print(f"  PID={os.getpid()} input={airr.name}", flush=True)

        for i, row in enumerate(reader, 1):
            rec = next(fq_iter, None)
            if rec is None:
                raise ValueError(f"FASTQ ended early: {sample} AIRR row {i}")

            fq_id = fastq_id(rec[0])
            fq_seq = rec[1].strip().upper()

            if fq_id != row["sequence_id"]:
                raise ValueError(
                    f"ID/order mismatch {sample} row {i}: "
                    f"FASTQ={fq_id!r}, AIRR={row['sequence_id']!r}"
                )
            airr_seq = str(row.get("sequence", "")).upper()
            if expected_airr_sequence(fq_seq, row) != airr_seq:
                raise ValueError(
                    f"Sequence/orientation mismatch {sample} row {i}: {fq_id}"
                )

            reasons = evaluate(row, expected_locus)
            counts["input"] += 1

            if reasons:
                counts["rejected"] += 1
                reason_counts.update(reasons)
                rejected = dict(row)
                rejected["filter_reasons"] = ";".join(reasons)
                reject_writer.writerow(rejected)
            else:
                counts["passed"] += 1
                pass_writer.writerow(row)
                ofq.writelines(rec)

            now = time.monotonic()
            if now - last_heartbeat >= 30:
                elapsed = int(now - started)
                staging_mb = (pass_airr.stat().st_size + reject_airr.stat().st_size) / 1e6
                print(
                    f"  [{sample}] elapsed={elapsed}s rows={counts['input']:,} "
                    f"staging={staging_mb:.1f} MB PID={os.getpid()}",
                    flush=True,
                )
                last_heartbeat = now

        if next(fq_iter, None) is not None:
            raise ValueError(
                f"FASTQ has extra records after AIRR ended: {sample}"
            )

    return {
        "input": counts["input"],
        "passed": counts["passed"],
        "rejected": counts["rejected"],
        "pass_pct": (
            100.0 * counts["passed"] / counts["input"]
            if counts["input"] else 0.0
        ),
        "reasons": dict(reason_counts),
    }


In [ ]:
def _promote(staging, final):
    staging, final = Path(staging), Path(final)
    previous = final.parent / f".{final.name}.previous"

    if previous.exists():
        shutil.rmtree(previous)
    if final.exists():
        final.rename(previous)

    try:
        staging.rename(final)
    except Exception:
        if previous.exists() and not final.exists():
            previous.rename(final)
        raise

    if previous.exists():
        shutil.rmtree(previous)

def run_filter(force=FORCE):
    if FINAL_DIR.exists() and not force:
        raise FileExistsError(
            f"Output already exists: {FINAL_DIR}; "
            "set FORCE=True only for an intentional rebuild"
        )

    if STAGING_DIR.exists():
        if not force:
            raise FileExistsError(
                f"Staging output exists: {STAGING_DIR}; inspect/remove it "
                "or set FORCE=True to rebuild"
            )
        shutil.rmtree(STAGING_DIR)

    for subdir in ("fastq", "airr_pass", "airr_reject"):
        (STAGING_DIR / subdir).mkdir(parents=True, exist_ok=True)

    summary = {}
    for sample in SAMPLES:
        print(f"[filter] {sample}", flush=True)
        summary[sample] = filter_sample(sample, STAGING_DIR)
        s = summary[sample]
        print(
            f"  input={s['input']:,} passed={s['passed']:,} "
            f"({s['pass_pct']:.2f}%) rejected={s['rejected']:,}",
            flush=True,
        )

    qc = {
        "dataset": DATASET,
        "raw_source_dataset": RAW_SOURCE_DATASET,
        "organism": "Mus musculus",
        "preprocessing_branch": "fastp_q30_u40",
        "input": (
            "merged/fastq/*_assemble-pass.fastq.gz in exact lockstep with "
            "annotation/igblast/*.airr.tsv"
        ),
        "annotation_producer": "annotate_mouse.ipynb",
        "filters": {
            "min_vj_span_nt": MIN_VJ_LENGTH,
            "require_v_germline_start_eq_1": True,
            "require_complete_vdj_for_j_3prime_end": True,
            "require_expected_locus": EXPECTED_LOCUS,
            "require_v_call": True,
            "require_j_call": True,
            "require_productive": True,
            "reject_stop_codon": True,
            "min_v_identity_percent": MIN_V_IDENTITY,
            "min_j_identity_percent": MIN_J_IDENTITY,
            "max_v_support_evalue": MAX_V_SUPPORT,
            "max_j_support_evalue": MAX_J_SUPPORT,
        },
        "samples": summary,
    }

    (STAGING_DIR / "filter_summary.json").write_text(
        json.dumps(qc, indent=2) + "\n"
    )
    _promote(STAGING_DIR, FINAL_DIR)
    print("DONE", FINAL_DIR)
    return qc


## Входы и результаты

Требуются согласованные пары файлов для всех шести samples:

- `results/ERP003950/merged/fastq/*_assemble-pass.fastq.gz`;
- `results/ERP003950/annotation/igblast/*.airr.tsv`.

Результаты записываются в `results/ERP003950/post_annotation_filtered/`.


In [ ]:
qc = run_filter(force=FORCE)
qc
